# Planning de Staff pour les évenements étudiants

On désire gérer par des moyens informatiques (python) le planning de staff d’un événement BDE.
Pour un événement, il y a différents staff (ex: bar, sono, entrée …) et un certain nombre de staffeur.
Les staffs doivent tourner à chaque créneaux.

On souhaite également compter les heures de staff de chaques personne pour égaliser.

L’utilisateur choisit le timing de ses créneaux, il doit également pouvoir choisir combien de personne il veut pour chaque staff.

En entrée, le programme demande donc :
- **La plage horaire de l’event** (de 21h à 5h)
- **Le timing de roulement** (changement toutes les 1h / 2h)
- **Le nom et genre de chaque personne qui staff**
- **Leur contraintes** (ex: n’aime pas le vomi, le bar …)
- **Le nombre de personne qu’il doit y avoir pour chaque staff** (on choisira surement des valeurs par défaut pour certain staffs).

En sortie:
- Un excel général avec le planning créneau par créneau de tout le monde
- Un planning excel individuel
- Un planning image individuel (taille fond écran tel)


## 2nd temps: Contraintes suplémentaires
- veut staffer avec quelqu’un
- ne veut pas staffer avec quelqu’un
- dispo uniquement à certains créneaux

# Import et Setup

## Cellule à éxecuter pour travailler avec google collab

In [ ]:
# Librairies classiques
import pandas as pd
from datetime import datetime, timedelta
import random

# Google Sheets (via TON compte Google)
import gspread
from google.colab import auth
from google.auth import default

# AUTHENTIFICATION GOOGLE

auth.authenticate_user()

creds, _ = default()
gc = gspread.authorize(creds)

FOLDER_ID = "158zZyuL2fQ9_d5Ac2567Xf5OPOo8Qcno"

## Cellule à éxécuter pour travailler en local

In [9]:
# Librairies classiques
import pandas as pd
from datetime import datetime, timedelta
import random

## Définition des classes

In [10]:
##### Staff #####
class Staff:
  def __init__(self, nom, prenom, genre):
        self.nom = nom
        self.prenom = prenom
        self.genre = genre
        self.contrainte = []
        self.heures_staff = 0

  def __str__(self):
        return self.nom + " " + self.prenom

##### Poste #####
class Poste:
  def __init__(self, nom, nb_max_staffeur):
        self.nom = nom
        self.nb_max_staffeur = nb_max_staffeur

  def __str__(self):
        return self.nom

##### Créneau #####
class Creneau:
  def __init__(self, date_debut, date_fin):   # date = datetime(année, mois, jour, heure)
        self.date_debut = date_debut
        self.date_fin = date_fin

  def __str__(self):
        return f"Début: {self.date_debut.strftime("%-d/%m %Hh%M")}, Fin: {self.date_fin.strftime("%-d/%m %Hh%M")}"

  def duree(self):
        return self.date_fin - self.date_debut


##### Affectation #####
class Affectation:
    def __init__(self, staff, creneau, poste ):
        self.staff = staff
        self.creneau = creneau
        self.poste = poste

    def __str__(self):
        return f"{self.staff} --> {self.poste} --> {self.creneau}"


## Tests

In [11]:
# Un staffeur
s1 = Staff("Martin", "Jean", "H")
s2 = Staff("Dupont", "Michelle", "F")
s3 = Staff("Dupont", "Paul", "H")

# Un poste
p1 = Poste("Bar", 2)
p2 = Poste("Entrée", 1)

# Un créneau
c1 = datetime(2026, 3, 15, 20)
c2 = datetime(2026, 3, 16, 4)

In [12]:
print(Creneau(c1, c2))
print(Creneau(c1, c2).duree())

Début: 15/03 20h00, Fin: 16/03 04h00
8:00:00


In [13]:
a1 = Affectation(s1, Creneau(c1, c2), p1)
print(a1)

Martin Jean --> Bar --> Début: 15/03 20h00, Fin: 16/03 04h00


# Fonctions principales
## Génération automatique des créneaux

In [14]:
def generer_creneaux(date_debut, date_fin, intervalle):
  creneaux=[]
  while date_debut < date_fin:
    creneaux.append(Creneau(date_debut, date_debut + timedelta(hours=intervalle)))
    date_debut = date_debut + timedelta(hours=intervalle)

  return creneaux

for c in generer_creneaux(c1, c2, 1):
  print(c)

Début: 15/03 20h00, Fin: 15/03 21h00
Début: 15/03 21h00, Fin: 15/03 22h00
Début: 15/03 22h00, Fin: 15/03 23h00
Début: 15/03 23h00, Fin: 16/03 00h00
Début: 16/03 00h00, Fin: 16/03 01h00
Début: 16/03 01h00, Fin: 16/03 02h00
Début: 16/03 02h00, Fin: 16/03 03h00
Début: 16/03 03h00, Fin: 16/03 04h00


## Saisie des informations des staffeurs, des postes, de l'event



In [15]:
def saisir_staffeurs():
    liste_staffeurs = []
    while True:
      s = Staff(input("nom: "), input("prenom: "), input("genre: (H/F) "))
      liste_staffeurs.append(s)

      encore = input("encore? (O/N) ")
      if encore == "N":
        break

    return liste_staffeurs

In [16]:
def saisir_postes():
    liste_postes = []
    while True:
      p = Poste(input("nom: "), int(input("nombre max de staffeurs: ")))
      liste_postes.append(p)

      encore = input("encore? (O/N) ")
      if encore == "N":
        break

    return liste_postes

In [17]:
def event():
  date_debut = datetime.strptime(input("Date et heure de début: (dd/mm/YYYY  HHhMM)"),"%d/%m/%Y %Hh%M")
  date_fin = datetime.strptime(input("Date et heure de fin: (dd/mm/YYYY  HHhMM)"),"%d/%m/%Y %Hh%M")

  intervalle = int(input("Durée d'un créneau: "))

  return generer_creneaux(date_debut, date_fin, intervalle)

## Données de test

In [18]:
staffeurs = [
    Staff("Martin", "Jean", "H"),
    Staff("Dupont", "Michelle", "F"),
    Staff("Bernard", "Paul", "H"),
    Staff("Petit", "Sophie", "F"),
    Staff("Durand", "Lucas", "H"),
    Staff("Moreau", "Emma", "F"),
]

postes = [
    Poste("Bar", 2),
    Poste("Entrée", 1),
    Poste("Sono", 1),
]

creneaux = generer_creneaux(datetime(2026, 4, 3, 21), datetime(2026, 4, 4, 5), 1)

# Algorithme d'affectation

1. Sélectionner un créneau (horaire + poste)
2. Sélectionner la personne qui a le moins d'heure de staff (tri)
3. Vérifier si la personne est disponible, sinon sélectionner une autre
4. Affecter cette personne au créneau
5. Recommencer pour chaque poste puis pour les heures suivantes

In [19]:
def affecter(staffeurs, postes, creneaux):
    affectations = []

    for creneau in creneaux:
        poste = postes.copy()
        random.shuffle(poste)
        for p in poste:

          for place in range(p.nb_max_staffeur):

              # 1. Trier les staffeurs par heures_staff
              sorted_staffeurs = sorted(staffeurs, key=lambda s: s.heures_staff)

              # 2. Trouver le premier disponible et Créer l'affectation
              for staff in sorted_staffeurs:
                if not any(a.staff == staff and a.creneau == creneau for a in affectations):
                  affectations.append(Affectation(staff, creneau, p))
                  sorted_staffeurs.remove(staff)

                # 3. Mettre à jour heures_staff
                  staff.heures_staff += creneau.duree().seconds//3600
                  break


    return affectations

### Test


In [20]:
for a in affecter(staffeurs, postes, creneaux):
    print(a)

affectations = affecter(staffeurs, postes, creneaux)
for s in staffeurs:
    print(s, s.heures_staff, "h")

Martin Jean --> Bar --> Début: 3/04 21h00, Fin: 3/04 22h00
Dupont Michelle --> Bar --> Début: 3/04 21h00, Fin: 3/04 22h00
Bernard Paul --> Entrée --> Début: 3/04 21h00, Fin: 3/04 22h00
Petit Sophie --> Sono --> Début: 3/04 21h00, Fin: 3/04 22h00
Durand Lucas --> Bar --> Début: 3/04 22h00, Fin: 3/04 23h00
Moreau Emma --> Bar --> Début: 3/04 22h00, Fin: 3/04 23h00
Martin Jean --> Entrée --> Début: 3/04 22h00, Fin: 3/04 23h00
Dupont Michelle --> Sono --> Début: 3/04 22h00, Fin: 3/04 23h00
Bernard Paul --> Bar --> Début: 3/04 23h00, Fin: 4/04 00h00
Petit Sophie --> Bar --> Début: 3/04 23h00, Fin: 4/04 00h00
Durand Lucas --> Sono --> Début: 3/04 23h00, Fin: 4/04 00h00
Moreau Emma --> Entrée --> Début: 3/04 23h00, Fin: 4/04 00h00
Martin Jean --> Bar --> Début: 4/04 00h00, Fin: 4/04 01h00
Dupont Michelle --> Bar --> Début: 4/04 00h00, Fin: 4/04 01h00
Bernard Paul --> Sono --> Début: 4/04 00h00, Fin: 4/04 01h00
Petit Sophie --> Entrée --> Début: 4/04 00h00, Fin: 4/04 01h00
Durand Lucas --> Son

## Création des excel

In [23]:
pip install aspose-cells-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 MB 508.0 kB/s  0:02:36m0:00:0100:04
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from aspose import pycore
from aspose.cells import Workbook, SaveFormat, FileFormatType

# Create Workbook object.
workbook = Workbook()

# Access the first worksheet of the workbook.
worksheet = workbook.worksheets[0]

# Get the desired cell(s) of the worksheet and input the value into the cell(s).
worksheet.cells.get("A1").put_value("ColumnA")
worksheet.cells.get("B1").put_value("ColumnB")
worksheet.cells.get("A2").put_value("ValueA")
worksheet.cells.get("B2").put_value("ValueB")

# Save the workbook as EXCEL file.
workbook.save("output.xlsx")